# E1: FLAN-T5 Sentence No-context Fine-tuning

**Project:** CLEF SimpleText 2026 - Task 1.1 Scientific Text Simplification

This notebook fine-tunes `google/flan-t5-base` for sentence-level biomedical text simplification without context.

## Environment Setup

Run this cell once in a fresh Jupyter/RunPod environment to install the project dependencies into the active kernel.


In [ ]:
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

requirements_path = project_root / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(f"Could not find requirements.txt at {requirements_path}")

# RunPod usually ships with CUDA PyTorch preinstalled. Do not install torch here;
# requirements.txt intentionally leaves torch out to preserve the CUDA stack.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--no-cache-dir",
    "-r",
    str(requirements_path),
])


## 1. Imports and Configuration

The experiment uses only the complex sentence as input. No neighbouring sentences, paragraph context, document context, `doc_pos`, `doc_quint`, or `doc_len` are used.

In [ ]:
from __future__ import annotations

import gc
import os
import random
import time
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

In [ ]:
SEED = 42
MODEL_NAME = "google/flan-t5-base"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context"
TRAIN_PATH = DATA_DIR / "train_clean.csv"
VAL_PATH = DATA_DIR / "val_clean.csv"
TEST_PATH = DATA_DIR / "test_clean.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "flan_t5_sentence_no_context"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "flan_t5_sentence_no_context_predictions.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fp16_enabled = torch.cuda.is_available()

print(f"Project root: {PROJECT_ROOT}")
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"fp16 enabled: {fp16_enabled}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Prediction path: {PREDICTION_PATH}")

## 2. Load Data

The notebook reads the preprocessed no-context sentence-level splits with `complex` as input and `simple` as the reference target.

In [ ]:
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} split: {path}")
    df = pd.read_csv(path)
    expected_columns = ["pair_id", "sent_id", "label", "complex", "simple"]
    missing = [column for column in expected_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{split_name} split is missing columns: {missing}")
    df = df[expected_columns].copy()
    df["complex"] = df["complex"].fillna("").astype(str).str.strip()
    df["simple"] = df["simple"].fillna("").astype(str).str.strip()
    return df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)

train_df = load_split(TRAIN_PATH, "train")
val_df = load_split(VAL_PATH, "validation")
test_df = load_split(TEST_PATH, "test")

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name}: {len(df):,} rows")

print()
print("Column names:")
print(train_df.columns.tolist())

print()
print("Missing values:")
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print()
    print(name)
    print(df.isna().sum())


## 3. Data Inspection

We inspect examples and basic length statistics. Based on these statistics, this experiment uses `max_source_length = 256` and `max_target_length = 128`.

In [ ]:
display(train_df.head())

def word_count(text: str) -> int:
    return len(str(text).split())

length_frames = []
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    temp = pd.DataFrame({
        "split": split_name,
        "complex_words": df["complex"].map(word_count),
        "simple_words": df["simple"].map(word_count),
    })
    length_frames.append(temp)

length_df = pd.concat(length_frames, ignore_index=True)
length_summary = (
    length_df.groupby("split")[["complex_words", "simple_words"]]
    .agg(["mean", "median", "min", "max", lambda values: values.quantile(0.95)])
    .round(2)
)
length_summary.columns = [f"{col}_{stat if isinstance(stat, str) else 'p95'}" for col, stat in length_summary.columns]
length_summary

In [ ]:
max_source_length = 256
max_target_length = 128

print(f"max_source_length = {max_source_length}")
print(f"max_target_length = {max_target_length}")

## 4. Tokenization

Each input is formatted as `simplify: {complex}`. The target is the `simple` sentence. Padding token ids in labels are replaced with `-100` so they are ignored by the loss.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

PREFIX = "simplify: "

def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [PREFIX + str(text) for text in examples["complex"]]
    targets = [str(text) for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs

## 5. Dataset Creation

The Hugging Face `Dataset` objects keep only the no-context columns needed for training plus ids used later for saving predictions.

In [ ]:
train_dataset_raw = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset_raw = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset_raw = Dataset.from_pandas(test_df, preserve_index=False)

remove_columns = train_dataset_raw.column_names
train_dataset = train_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
val_dataset = val_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
test_dataset = test_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

print(train_dataset)
print(val_dataset)
print(test_dataset)

## 6. Model Loading

The model is loaded from `google/flan-t5-base` and moved to GPU when CUDA is available.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(f"Model loaded on: {next(model.parameters()).device}")

## 7. Training

The training run uses `Seq2SeqTrainer` with early stopping. If GPU memory is insufficient, reduce `per_device_train_batch_size` to `4` and set `gradient_accumulation_steps = 2`.

In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs,
        )
    except TypeError:
        return Seq2SeqTrainingArguments(
            eval_strategy="epoch",
            **base_kwargs,
        )

training_args = build_training_args()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "best_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best_model"))

### Training Log

Use this table to inspect training and validation results after each logging step and epoch.


In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])


## 8. Inference on Test Set

Predictions are generated for all test examples with beam search. Only the `complex` sentence is used to create model inputs.

In [ ]:
def generate_predictions(df: pd.DataFrame) -> pd.DataFrame:
    predictions = []
    model.eval()

    for sentence in tqdm(df["complex"], total=len(df), desc="Generating test predictions"):
        input_text = PREFIX + str(sentence)
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            max_length=max_source_length,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                early_stopping=True,
            )
        prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        predictions.append(prediction)

    output_df = df[["pair_id", "sent_id", "label", "complex", "simple"]].copy()
    output_df["prediction"] = predictions
    return output_df

prediction_df = generate_predictions(test_df)
prediction_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH}")
prediction_df.head()

## 9. Evaluation

The notebook reports SARI, BLEU, and BERTScore. It tries `evaluate` for SARI/BLEU when available, with local fallbacks to keep the experiment runnable in offline or restricted environments. BERTScore is computed with the `bert-score` package.

In [ ]:
def get_ngrams(tokens: list[str], n: int) -> Counter[tuple[str, ...]]:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def safe_f1(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def sari_sentence(source: str, prediction: str, reference: str, max_ngram: int = 4) -> float:
    source_tokens = source.lower().split()
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    add_scores = []
    keep_scores = []
    delete_scores = []

    for n in range(1, max_ngram + 1):
        source_ngrams = set(get_ngrams(source_tokens, n))
        pred_ngrams = set(get_ngrams(pred_tokens, n))
        ref_ngrams = set(get_ngrams(ref_tokens, n))

        add_pred = pred_ngrams - source_ngrams
        add_ref = ref_ngrams - source_ngrams
        add_precision = len(add_pred & add_ref) / len(add_pred) if add_pred else 0.0
        add_recall = len(add_pred & add_ref) / len(add_ref) if add_ref else 0.0
        add_scores.append(safe_f1(add_precision, add_recall))

        keep_pred = pred_ngrams & source_ngrams
        keep_ref = ref_ngrams & source_ngrams
        keep_precision = len(keep_pred & keep_ref) / len(keep_pred) if keep_pred else 0.0
        keep_recall = len(keep_pred & keep_ref) / len(keep_ref) if keep_ref else 0.0
        keep_scores.append(safe_f1(keep_precision, keep_recall))

        delete_pred = source_ngrams - pred_ngrams
        delete_ref = source_ngrams - ref_ngrams
        delete_precision = len(delete_pred & delete_ref) / len(delete_pred) if delete_pred else 0.0
        delete_scores.append(delete_precision)

    return 100 * float(np.mean([np.mean(add_scores), np.mean(keep_scores), np.mean(delete_scores)]))


def compute_sari_score(sources: list[str], predictions: list[str], references: list[str]) -> float:
    if evaluate is not None:
        for metric_name in ["sari", "evaluate-metric/sari"]:
            try:
                sari_metric = evaluate.load(metric_name)
                result = sari_metric.compute(
                    sources=sources,
                    predictions=predictions,
                    references=[[reference] for reference in references],
                )
                return float(result["sari"])
            except Exception:
                pass
    scores = [
        sari_sentence(source, prediction, reference)
        for source, prediction, reference in zip(sources, predictions, references, strict=True)
    ]
    return float(np.mean(scores))


def compute_bleu_score(predictions: list[str], references: list[str]) -> float:
    if evaluate is not None:
        try:
            bleu_metric = evaluate.load("bleu")
            result = bleu_metric.compute(
                predictions=predictions,
                references=[[reference] for reference in references],
            )
            return float(result["bleu"])
        except Exception:
            pass
    return float(sacrebleu.corpus_bleu(predictions, [references]).score)


def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    valid_df = df.copy()
    valid_df["prediction"] = valid_df["prediction"].fillna("").astype(str)
    valid_df = valid_df[valid_df["prediction"].str.strip().ne("")].reset_index(drop=True)

    sources = valid_df["complex"].tolist()
    predictions = valid_df["prediction"].tolist()
    references = valid_df["simple"].tolist()

    sari_score = compute_sari_score(sources, predictions, references)
    bleu_score = compute_bleu_score(predictions, references)
    bert_precision, bert_recall, bert_f1 = bert_score(
        predictions,
        references,
        lang="en",
        verbose=False,
    )

    return pd.DataFrame(
        [
            {"metric": "SARI", "score": sari_score},
            {"metric": "BLEU", "score": bleu_score},
            {"metric": "BERTScore Precision", "score": float(bert_precision.mean())},
            {"metric": "BERTScore Recall", "score": float(bert_recall.mean())},
            {"metric": "BERTScore F1", "score": float(bert_f1.mean())},
        ]
    )

metrics_summary = compute_metrics(prediction_df)
metrics_summary

## 10. Qualitative Analysis

Inspect random examples with the source sentence, reference simplification, and FLAN-T5 prediction.

In [ ]:
example_columns = ["complex", "simple", "prediction"]
prediction_df[example_columns].sample(n=min(10, len(prediction_df)), random_state=SEED)

## 11. Experiment Summary

This notebook implements E1: fine-tuning `google/flan-t5-base` for CLEF SimpleText 2026 Task 1.1 sentence-level scientific text simplification.

- **Task:** biomedical sentence simplification
- **Model:** `google/flan-t5-base`
- **Input:** `simplify: {complex}`
- **Target:** `{simple}`
- **Context setting:** no context; only the complex sentence is used
- **Dataset sizes:** train 6742, validation 984, test 892
- **Training setup:** 5 epochs, learning rate 5e-5, batch size 8, epoch evaluation/saving, early stopping patience 2
- **Evaluation metrics:** SARI, BLEU, BERTScore
- **Comparison target:** E0 Llama 3.1 8B Instruct zero-shot baseline: SARI 28.25, BLEU 0.036, BERTScore F1 0.897

Context-aware modelling and plan-guided modelling are intentionally left for later experiments.